In [2]:
from pathlib import Path
from datetime import datetime
import math

import pandas as pd
import requests


# ============================================================
# CONFIGURATION
# ============================================================

DATA_DIR = Path("data")

NORMALS_FILE = DATA_DIR / "climate_normals.csv"
STATIONS_FILE = DATA_DIR / "climate_stations.csv"

# ECCC 1991-2020 station inventory
STATION_INVENTORY_URL = (
    "https://climate.weather.gc.ca/"
    "climate_normals/station_inventory_e.html?yr=1991"
)


# ============================================================
# DOWNLOAD ECCC STATION INVENTORY
# ============================================================

def download_station_inventory():
    """
    Download the ECCC station inventory.

    The inventory is downloaded only if the local file does
    not already exist.
    """

    if STATIONS_FILE.exists():

        print(
            "Loading cached ECCC station inventory..."
        )

        return pd.read_csv(STATIONS_FILE)

    print(
        "Downloading ECCC station inventory..."
    )

    DATA_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    response = requests.get(
        STATION_INVENTORY_URL,
        timeout=30
    )

    response.raise_for_status()

    STATIONS_FILE.write_bytes(
        response.content
    )

    print(
        f"Saved ECCC station inventory to "
        f"{STATIONS_FILE}"
    )

    return pd.read_csv(
        STATIONS_FILE
    )


# ============================================================
# LOAD CLIMATE DATA
# ============================================================

def load_data():

    if not NORMALS_FILE.exists():

        raise FileNotFoundError(
            "\nCould not find the climate normals file:\n"
            f"    {NORMALS_FILE}\n\n"
            "Make sure your ECCC normals CSV is named "
            "'climate_normals.csv' and is inside the "
            "'data' directory."
        )

    print(
        "Loading climate normals..."
    )

    normals = pd.read_csv(
        NORMALS_FILE
    )

    stations = download_station_inventory()

    return normals, stations


# ============================================================
# NORMALIZE STATION NAMES
# ============================================================

def normalize_station_name(name):
    """
    Normalize ECCC station names so that small differences
    between the normals dataset and station inventory do not
    prevent matching.

    For example:

        VANCOUVER HARBOUR CS
        VANCOUVER HARBOUR

    both become:

        VANCOUVER HARBOUR
    """

    name = str(name).upper().strip()

    # Common ECCC suffixes
    suffixes = [
        " CS",
        " AUTO",
        " AUT",
        " A",
        " CDA",
        " STATION"
    ]

    changed = True

    while changed:

        changed = False

        for suffix in suffixes:

            if name.endswith(suffix):

                name = (
                    name[:-len(suffix)]
                    .strip()
                )

                changed = True

    return name


# ============================================================
# PARSE ECCC COORDINATES
# ============================================================

def parse_coordinate(value):
    """
    Convert an ECCC coordinate to decimal degrees.

    Handles:

        49.2827

    and:

        49°16'57.720"N

    """

    if pd.isna(value):
        return None

    value = str(value).strip()

    # --------------------------------------------------------
    # Already decimal
    # --------------------------------------------------------

    try:

        return float(value)

    except ValueError:

        pass

    # --------------------------------------------------------
    # DMS format
    # --------------------------------------------------------

    value = (
        value
        .replace("°", " ")
        .replace("'", " ")
        .replace('"', " ")
    )

    parts = value.split()

    if len(parts) < 3:
        return None

    try:

        degrees = float(parts[0])
        minutes = float(parts[1])
        seconds = float(parts[2])

    except ValueError:

        return None

    result = (
        abs(degrees)
        + minutes / 60
        + seconds / 3600
    )

    # South and West are negative

    if (
        "W" in value.upper()
        or "S" in value.upper()
    ):

        result *= -1

    return result


# ============================================================
# PREPARE STATION DATA
# ============================================================

def prepare_stations(stations):

    columns = {
        column.upper(): column
        for column in stations.columns
    }

    def find_column(*names):

        for name in names:

            if name.upper() in columns:

                return columns[
                    name.upper()
                ]

        raise KeyError(
            "\nCould not find any of "
            f"{names} in the ECCC station "
            "inventory.\n\n"
            f"Available columns:\n"
            f"{list(stations.columns)}"
        )

    # --------------------------------------------------------
    # Find relevant columns
    # --------------------------------------------------------

    name_col = find_column(
        "LOCATION_NAME",
        "STATION_NAME",
        "COMPOSITE_STATION_NAME"
    )

    lat_col = find_column(
        "LATITUDE",
        "LAT"
    )

    lon_col = find_column(
        "LONGITUDE",
        "LON",
        "LONG"
    )

    stations = stations.copy()

    # --------------------------------------------------------
    # Standardized station name
    # --------------------------------------------------------

    stations["LOCATION_NAME"] = (
        stations[name_col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    stations["MATCH_NAME"] = (
        stations["LOCATION_NAME"]
        .apply(
            normalize_station_name
        )
    )

    # --------------------------------------------------------
    # Convert coordinates to decimal degrees
    # --------------------------------------------------------

    stations["LATITUDE_DECIMAL"] = (
        stations[lat_col]
        .apply(parse_coordinate)
    )

    stations["LONGITUDE_DECIMAL"] = (
        stations[lon_col]
        .apply(parse_coordinate)
    )

    # --------------------------------------------------------
    # Remove stations without coordinates
    # --------------------------------------------------------

    stations = stations.dropna(
        subset=[
            "LATITUDE_DECIMAL",
            "LONGITUDE_DECIMAL"
        ]
    )

    print(
        f"Loaded {len(stations):,} "
        "ECCC stations with coordinates."
    )

    return stations


# ============================================================
# HAVERSINE DISTANCE
# ============================================================

def haversine_distance(
    lat1,
    lon1,
    lat2,
    lon2
):
    """
    Calculate the distance between two latitude/longitude
    coordinates in kilometres.
    """

    earth_radius = 6371.0

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)

    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return earth_radius * c


# ============================================================
# FIND NEAREST STATION
# ============================================================

def find_nearest_station(
    lat,
    lon,
    stations
):
    """
    Find the nearest ECCC climate-normal station.

    Uses a fast geographic approximation to identify the
    closest station, then calculates the actual distance
    using the Haversine formula.
    """

    # --------------------------------------------------------
    # Fast approximation
    # --------------------------------------------------------

    longitude_scale = math.cos(
        math.radians(lat)
    )

    distances = (
        (
            stations["LATITUDE_DECIMAL"]
            - lat
        ) ** 2
        +
        (
            (
                stations["LONGITUDE_DECIMAL"]
                - lon
            )
            * longitude_scale
        ) ** 2
    )

    index = distances.idxmin()

    station = stations.loc[
        index
    ].copy()

    # --------------------------------------------------------
    # Calculate actual distance
    # --------------------------------------------------------

    station["distance_km"] = (
        haversine_distance(
            lat,
            lon,
            station["LATITUDE_DECIMAL"],
            station["LONGITUDE_DECIMAL"]
        )
    )

    return station


# ============================================================
# GET CLIMATE NORMAL
# ============================================================

def get_weather(
    lat,
    lon,
    normals,
    stations
):
    """
    Get the ECCC 1991-2020 climate normal for the nearest
    station.

    The normal returned is for the current month.

    Returns:

        station
        distance_km
        month
        temperature
        temperature_max
        temperature_min
        latitude
        longitude
    """

    # --------------------------------------------------------
    # Find nearest station
    # --------------------------------------------------------

    station = find_nearest_station(
        lat,
        lon,
        stations
    )

    station_name = station[
        "LOCATION_NAME"
    ]

    match_name = normalize_station_name(
        station_name
    )

    # --------------------------------------------------------
    # Normalize names in the normals dataset
    # --------------------------------------------------------

    normals_match = (
        normals["LOCATION_NAME"]
        .astype(str)
        .str.strip()
        .str.upper()
        .apply(
            normalize_station_name
        )
    )

    station_normals = normals[
        normals_match == match_name
    ]

    # --------------------------------------------------------
    # Check whether a matching station exists
    # --------------------------------------------------------

    if station_normals.empty:

        raise ValueError(
            "\nNo climate normals found for "
            f"station '{station_name}'.\n"
            f"Normalized name: '{match_name}'"
        )

    # --------------------------------------------------------
    # Current month
    # --------------------------------------------------------

    month = datetime.now().strftime(
        "%b"
    )

    # --------------------------------------------------------
    # Get a particular normal
    # --------------------------------------------------------

    def get_element(element):

        result = station_normals.loc[
            station_normals[
                "NORMALS_ELEMENT"
            ] == element,
            month
        ]

        if result.empty:

            return None

        value = result.iloc[0]

        if (
            pd.isna(value)
            or str(value).strip() == ""
        ):

            return None

        return float(value)

    # --------------------------------------------------------
    # Return result
    # --------------------------------------------------------

    return {

        "station": station_name,

        "distance_km": round(
            station["distance_km"],
            1
        ),

        "month": month,

        "temperature": get_element(
            "Daily Average (°C)"
        ),

        "temperature_max": get_element(
            "Daily Maximum (°C)"
        ),

        "temperature_min": get_element(
            "Daily Minimum (°C)"
        ),

        "latitude": station[
            "LATITUDE_DECIMAL"
        ],

        "longitude": station[
            "LONGITUDE_DECIMAL"
        ]
    }


# ============================================================
# LOAD EVERYTHING
# ============================================================

normals, stations = load_data()

stations = prepare_stations(
    stations
)


# ============================================================
# TEST
# ============================================================

if __name__ == "__main__":

    print()
    print(
        "Testing Vancouver..."
    )

    weather = get_weather(
        49.2827,
        -123.1207,
        normals,
        stations
    )

    print()
    print(
        "Climate Normal"
    )
    print(
        "---------------------------"
    )

    print(
        f"Station:       "
        f"{weather['station']}"
    )

    print(
        f"Distance:      "
        f"{weather['distance_km']} km"
    )

    print(
        f"Month:         "
        f"{weather['month']}"
    )

    print(
        f"Average:       "
        f"{weather['temperature']} °C"
    )

    print(
        f"Average high:  "
        f"{weather['temperature_max']} °C"
    )

    print(
        f"Average low:   "
        f"{weather['temperature_min']} °C"
    )

Loading climate normals...
Loading cached ECCC station inventory...
Loaded 938 ECCC stations with coordinates.

Testing Vancouver...


ValueError: 
No climate normals found for station 'VANCOUVER HARBOUR CS'.
Normalized name: 'VANCOUVER HARBOUR'